# CEREBRO Evidence Chain

CEREBRO's knowledge fragment can always be traced to an identifiable, unchanged source artifact and an exact location inside it.

``` mermaid
flowchart TD
    U[USER-001]
    A[ART-0001]
    F[Knowledge Fragment KF-0001]
    L[Exact Source Location]
    S[Original Artifact]
    H[SHA-256 Fingerprint]

    U -->|owns / ingests| A
    A -->|contains| F
    F -->|references| L
    L -->|resolves to| S
    A -->|verified by| H
```

## EXP-GT-001 — Source Evidence Integrity

### Hypothesis

CEREBRO can uniquely identify a source artifact and maintain a
resolvable provenance path from a knowledge fragment to the original
source evidence.

### Controlled Variables

- Artifact ID: `ART-0001`
- Owner: `USER-001`
- Modality: Text
- Dataset: `CEREBRO-GT-v0.1`
- Ground truth: Manually verified

In [1]:
from pathlib import Path
import hashlib

source_path = Path("../data/raw/text/benchmark_001.txt")

assert source_path.exists(), "Source artifact cannot be resolved."
assert source_path.stat().st_size > 0, "Source artifact is empty."

source_bytes = source_path.read_bytes()
source_text = source_path.read_text(encoding="utf-8")
sha256 = hashlib.sha256(source_bytes).hexdigest()

print("Artifact : ART-0001")
print("Path     :", source_path)
print("Bytes    :", len(source_bytes))
print("SHA-256  :", sha256)
print("\nContent:")
print(source_text)

Artifact : ART-0001
Path     : ../data/raw/text/benchmark_001.txt
Bytes    : 294
SHA-256  : 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832

Content:
CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge. It maintains provenance between knowledge fragments and their original source artifacts. The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### Upgrade ART-0001 from placeholder to real artifact

In [2]:
artifact_001 = {
    "artifact_id": "ART-0001",
    "owner_id": "USER-001",

    "title": "CEREBRO Benchmark Text",
    "modality": "text",
    "mime_type": "text/plain",
    "language": "en",

    "authors": [
        {
            "author_id": "PERSON-001",
            "name": "Benchmark Author"
        }
    ],

    "source": {
        "filename": source_path.name,
        "storage_uri": str(source_path),
        "sha256": sha256
    },

    "benchmark": {
        "dataset": "CEREBRO-GT-v0.1",
        "manually_verified": True
    }
}

artifact_001

{'artifact_id': 'ART-0001',
 'owner_id': 'USER-001',
 'title': 'CEREBRO Benchmark Text',
 'modality': 'text',
 'mime_type': 'text/plain',
 'language': 'en',
 'authors': [{'author_id': 'PERSON-001', 'name': 'Benchmark Author'}],
 'source': {'filename': 'benchmark_001.txt',
  'storage_uri': '../data/raw/text/benchmark_001.txt',
  'sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832'},
 'benchmark': {'dataset': 'CEREBRO-GT-v0.1', 'manually_verified': True}}

### Add the first exact ground-truth fragment

In [4]:
expected_fragment = (
    "It maintains provenance between knowledge fragments "
    "and their original source artifacts."
)

start_char = source_text.index(expected_fragment)
end_char = start_char + len(expected_fragment)

knowledge_fragment_001 = {
    "fragment_id": "KF-0001",
    "artifact_id": "ART-0001",
    "owner_id": "USER-001",

    "content": expected_fragment,

    "source_location": {
        "type": "text_span",
        "start_char": start_char,
        "end_char": end_char
    },

    "provenance": {
        "extraction_method": "manual_ground_truth",
        "parent_artifact_id": "ART-0001",
        "source_sha256": sha256
    },

    "manually_verified": True
}

knowledge_fragment_001

{'fragment_id': 'KF-0001',
 'artifact_id': 'ART-0001',
 'owner_id': 'USER-001',
 'content': 'It maintains provenance between knowledge fragments and their original source artifacts.',
 'source_location': {'type': 'text_span', 'start_char': 86, 'end_char': 174},
 'provenance': {'extraction_method': 'manual_ground_truth',
  'parent_artifact_id': 'ART-0001',
  'source_sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832'},
 'manually_verified': True}

### Perform round-trip test

In [5]:
location = knowledge_fragment_001["source_location"]

recovered_text = source_text[
    location["start_char"]:
    location["end_char"]
]

assert recovered_text == knowledge_fragment_001["content"]

print("✓ Artifact resolved")
print("✓ Source location resolved")
print("✓ Knowledge reconstructed from source")
print("✓ Content matches ground truth")
print("✓ Provenance chain valid")

✓ Artifact resolved
✓ Source location resolved
✓ Knowledge reconstructed from source
✓ Content matches ground truth
✓ Provenance chain valid



''' mermaid
flowchart TD

    %% =========================
    %% SOURCE
    %% =========================
    U[User / Knowledge Twin Owner]
    AU[Author / Person / Organization]
    A[Original Artifact]
    REG[Artifact Registry]

    U -->|owns / ingests| A
    AU -->|authored / contributed| A
    A --> REG

    %% =========================
    %% INGESTION
    %% =========================
    REG --> ID[Modality Identification]

    ID --> TXT[Text Processing]
    ID --> PDF[PDF Processing]
    ID --> IMG[Image Processing]
    ID --> AUD[Audio Processing]
    ID --> VID[Video Processing]

    TXT --> EXT[Extracted Representation]
    PDF --> EXT
    IMG --> EXT
    AUD --> EXT
    VID --> EXT

    %% =========================
    %% KNOWLEDGE CONSTRUCTION
    %% =========================
    EXT --> N[Normalization]
    N --> C[Chunking]
    C --> KF[Knowledge Fragments]

    KF --> KE[Knowledge Extraction]
    KF --> EMB[Embedding]

    KE --> KG[Knowledge Graph]
    EMB --> VS[Vector Stores]

    %% =========================
    %% RECALL
    %% =========================
    Q[User Query / Recall Cue]

    Q --> RET[Retrieval]
    VS --> RET
    KG --> RET

    RET --> F[Fusion / Reranking]
    F --> AR[Assisted Recollection]
    F --> GAL[Galaxy]

    %% =========================
    %% SOURCE RESOLUTION
    %% =========================
    AR --> SR[Source Resolution]
    GAL --> SR
    SR --> A

    %% =========================
    %% EVALUATION
    %% =========================
    GT[Ground Truth]
    OUT[Pipeline Output]

    EXT -.-> OUT
    KF -.-> OUT
    KE -.-> OUT
    RET -.-> OUT
    F -.-> OUT

    GT --> EV[Evaluation Engine]
    OUT --> EV

    EV --> MET[Metrics / Scores]
    MET --> EXP[Experiment Record]

    EXP --> DEC{Architecture Decision?}
    DEC -->|Yes| ADR[ADR]
    DEC -->|No| NEXT[Next Experiment]
'''

### Real Evidence Test

## EXP-GT-001 — Source Evidence Integrity

### Hypothesis

CEREBRO can uniquely identify a source artifact and maintain a
resolvable provenance path from a knowledge fragment back to the
original source evidence.

### Controlled Variables

- Dataset: `CEREBRO-GT-v0.1`
- Artifact: `ART-0001`
- Owner: `USER-001`
- Modality: Text
- Ground Truth: Manually verified

### Resolve and fingerprint the physical artifact

storage path → where is it?

SHA-256      → is it the same artifact?

In [6]:
from pathlib import Path
import hashlib

source_path = Path("../data/raw/text/benchmark_001.txt")

assert source_path.exists(), "Source artifact cannot be resolved."
assert source_path.stat().st_size > 0, "Source artifact is empty."

source_bytes = source_path.read_bytes()
source_text = source_path.read_text(encoding="utf-8")

sha256 = hashlib.sha256(source_bytes).hexdigest()

print("Artifact : ART-0001")
print("File     :", source_path.name)
print("Bytes    :", len(source_bytes))
print("SHA-256  :", sha256)
print("\nContent:")
print(source_text)

Artifact : ART-0001
File     : benchmark_001.txt
Bytes    : 294
SHA-256  : 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832

Content:
CEREBRO is a Digital Knowledge Twin designed to preserve and connect human knowledge. It maintains provenance between knowledge fragments and their original source artifacts. The system supports assisted recollection by allowing users to navigate from knowledge back to its supporting evidence.


### Register the artifact

In [7]:
artifact_001 = {
    "artifact_id": "ART-0001",
    "owner_id": "USER-001",

    "title": "CEREBRO Benchmark Text",
    "modality": "text",
    "mime_type": "text/plain",
    "language": "en",

    "authors": [
        {
            "author_id": "PERSON-001",
            "name": "Benchmark Author"
        }
    ],

    "source": {
        "filename": source_path.name,
        "storage_uri": str(source_path),
        "sha256": sha256
    },

    "benchmark": {
        "dataset": "CEREBRO-GT-v0.1",
        "manually_verified": True
    }
}

artifact_001

{'artifact_id': 'ART-0001',
 'owner_id': 'USER-001',
 'title': 'CEREBRO Benchmark Text',
 'modality': 'text',
 'mime_type': 'text/plain',
 'language': 'en',
 'authors': [{'author_id': 'PERSON-001', 'name': 'Benchmark Author'}],
 'source': {'filename': 'benchmark_001.txt',
  'storage_uri': '../data/raw/text/benchmark_001.txt',
  'sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832'},
 'benchmark': {'dataset': 'CEREBRO-GT-v0.1', 'manually_verified': True}}

### Create a manually verified knowledge fragment

In [8]:
expected_fragment = (
    "It maintains provenance between knowledge fragments "
    "and their original source artifacts."
)

start_char = source_text.index(expected_fragment)
end_char = start_char + len(expected_fragment)

print("Start :", start_char)
print("End   :", end_char)
print("Text  :", source_text[start_char:end_char])

Start : 86
End   : 174
Text  : It maintains provenance between knowledge fragments and their original source artifacts.


### Register the fragment

In [9]:
knowledge_fragment_001 = {
    "fragment_id": "KF-0001",
    "artifact_id": "ART-0001",
    "owner_id": "USER-001",

    "content": expected_fragment,

    "source_location": {
        "type": "text_span",
        "start_char": start_char,
        "end_char": end_char
    },

    "provenance": {
        "extraction_method": "manual_ground_truth",
        "parent_artifact_id": "ART-0001",
        "source_sha256": sha256
    },

    "manually_verified": True
}

knowledge_fragment_001

{'fragment_id': 'KF-0001',
 'artifact_id': 'ART-0001',
 'owner_id': 'USER-001',
 'content': 'It maintains provenance between knowledge fragments and their original source artifacts.',
 'source_location': {'type': 'text_span', 'start_char': 86, 'end_char': 174},
 'provenance': {'extraction_method': 'manual_ground_truth',
  'parent_artifact_id': 'ART-0001',
  'source_sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832'},
 'manually_verified': True}

### Perform a rount-trip evidence test

In [10]:
location = knowledge_fragment_001["source_location"]

recovered_text = source_text[
    location["start_char"]:
    location["end_char"]
]

assert recovered_text == knowledge_fragment_001["content"]
assert knowledge_fragment_001["artifact_id"] == artifact_001["artifact_id"]
assert knowledge_fragment_001["provenance"]["source_sha256"] == artifact_001["source"]["sha256"]

print("✓ Artifact resolved")
print("✓ Artifact identity verified")
print("✓ Source location resolved")
print("✓ Knowledge reconstructed from original source")
print("✓ Content matches ground truth")
print("✓ Provenance chain valid")

✓ Artifact resolved
✓ Artifact identity verified
✓ Source location resolved
✓ Knowledge reconstructed from original source
✓ Content matches ground truth
✓ Provenance chain valid


### Process ends

``` Mermaid
flowchart LR
    KF[KF-0001] --> L[Character Span]
    L --> A[ART-0001]
    A --> F[benchmark_001.txt]
    F --> H[SHA-256 Verified]
    H --> E[Exact Evidence Recovered]
```